# QPE Error-Mitigation Analysis — Rigetti Cepheus-1-108Q

Turns the JSON files written by `checkRetrieve.py job_results_qpe` into
mitigated numbers with error bars.

This is the reduce half of the study. `error_mitig.py` and
`rigetti_qpuf_common.py` build circuits; `mitig_analysis.py` supplies the
estimators; this notebook is the driver that wires them to actual results.

**What it computes**

| cell | what it is |
|---|---|
| **raw** | λ=1, no correction — the baseline |
| **REM** | λ=1, readout channel inverted using `cal0`/`cal1` |
| **ZNE** | P_correct(λ) extrapolated to λ=0, no readout correction |
| **ZNE+REM** | REM applied at every λ, *then* extrapolated |

Primary observable is **P_correct** — the probability mass in the bin
`round(φ·2^n_prec)`. It is the expectation of a projector, hence linear in
the state, which is what makes extrapolating it legitimate. TVD and phase
error are reported alongside as consistency checks only; TVD is *not* linear
and must not be extrapolated.

**Prerequisites**

```bash
python submit_qpe.py            # Mitigation: both  ->  5 tasks
python checkRetrieve.py job_results_qpe
```

No hardware yet? Set `USE_SYNTHETIC = True` in §0 and the whole notebook runs
end to end on fabricated Braket-shaped counts. Do that first — it validates
the pipeline before you spend a reservation window on it.

In [ ]:
import glob
import json
import os
import sys
from datetime import datetime, timezone

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Make the module directory importable no matter where the kernel started
# (VS Code and Jupyter Lab disagree about the cwd, and a bare `import
# mitig_analysis` then fails with "No module named ...").
_MODDIR = next(
    (d for d in (os.getcwd(),
                 os.path.join(os.getcwd(), "braket_direct", "rigetti_cepheus_20260807"),
                 os.path.dirname(os.path.abspath("__file__")))
     if os.path.exists(os.path.join(d, "mitig_analysis.py"))),
    None,
)
if _MODDIR is None:
    raise RuntimeError(
        "Cannot find mitig_analysis.py. Start the kernel in "
        "braket_direct/rigetti_cepheus_20260807/, or set _MODDIR by hand."
    )
sys.path.insert(0, _MODDIR)

import mitig_analysis as ma
from rigetti_qpuf_common import (
    load_device_caps, transpile_for_rigetti, build_qpe_circuit,
    measured_physical_qubits,
)
try:                       # chiplet-aware placement; the submission path uses
    from rigetti_qpuf_common import place_and_route      # it when available
except ImportError:
    place_and_route = None

print(f"module dir : {_MODDIR}")

# -- Palette ---------------------------------------------------------------
# Categorical slots assigned to ENTITIES in a fixed order and never cycled,
# so a condition keeps its colour across every figure below. Validated for
# colour-vision deficiency on the light chart surface (worst adjacent pair
# ΔE 9.1 protan, 19.6 normal). Three of these sit under 3:1 contrast on the
# surface, so every figure also ships a printed table -- identity is never
# carried by colour alone.
C = {
    "ideal":   "#2a78d6",   # slot 1  blue
    "raw":     "#eb6834",   # slot 2  orange
    "rem":     "#1baf7a",   # slot 3  aqua
    "zne":     "#eda100",   # slot 4  yellow
    "zne_rem": "#e87ba4",   # slot 5  magenta
}
INK, INK_2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e3e2df", "#fcfcfb"

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "savefig.bbox": "tight",
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8984", "axes.linewidth": 0.8,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8,
    "text.color": INK, "axes.labelcolor": INK_2,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
    "legend.frameon": False,
})

PLOTS_DIR = os.path.join(_MODDIR, "plots")
os.makedirs(PLOTS_DIR, exist_ok=True)
print(f"figures -> {PLOTS_DIR}")

## 0. Configuration

In [ ]:
RESULTS_DIR = os.path.join(_MODDIR, "job_results_qpe")

# True  -> fabricate a complete REM+ZNE dataset in memory, touch no files.
#          Use this to exercise the notebook before the reservation window.
# False -> read RESULTS_DIR/*.json written by checkRetrieve.py.
USE_SYNTHETIC = False

# Pick a run family when the directory holds several. None -> the most recent.
# A prefix of (n_prec, n_targ, state_mode, phi, scales, ddd_sequence), so
# (5, 1, "known") pins a config and (5, 1, "known", 0.125, (1, 3, 5), "XY4")
# pins the DDD arm of it too.
SELECT_RUN = None

# "pooled"      -> sum counts across every batch of the selected family.
#                  3 x 10,000 shots becomes one 30,000-shot estimate.
# "<batch_id>"  -> analyse a single batch (see the list section 2 prints).
BATCH_MODE = "pooled"

# Only used when a run predates the `batch_id` field: the submission-time gap
# above which two tasks are considered separate invocations.
#
# Measured on the 2026-08-07 session data, which is about as clean a
# separation as this could be: the 5 tasks of one invocation were submitted
# 0.3-0.8 s apart (device.run() returns as soon as the task is created, it
# does not wait for execution), while consecutive invocations were 136 s
# apart. 30 s sits ~37x above the within-batch gap and ~4.5x below the
# between-batch one. The first default tried here was 180 s, which silently
# merged two batches into one -- if this number is too LARGE the failure is
# invisible, so err small.
BATCH_GAP_S = 30

N_BOOT     = 500                 # bootstrap resamples
REM_METHOD = "simplex"           # "simplex" (constrained, default) | "naive"
ZNE_MODELS = ("richardson", "linear", "exponential")
SEED       = 0

print(f"results dir   : {RESULTS_DIR}")
print(f"synthetic     : {USE_SYNTHETIC}")
print(f"bootstrap     : {N_BOOT} resamples, REM method '{REM_METHOD}'")

## 1. Synthetic dataset (optional)

Fabricates exactly what `checkRetrieve.py` would have written: five payload
dicts (λ ∈ {1,3,5} plus `cal0`/`cal1`) whose `counts` are Braket-shaped —
**one character per measured qubit, ordered by ascending physical index**.

The `clbit_to_phys` map here is deliberately *scrambled* (`[37, 12, 45, 20, 3]`)
rather than sorted. That is the point: if the notebook silently assumed
"clbit *k* is the *k*-th smallest physical qubit" it would decode a different,
plausible-looking, wrong distribution. §3 demonstrates that failure explicitly.

Ground truth baked in — REM should recover roughly these:
`P_correct(λ=1) ≈ 0.48`, per-qubit readout error ≈ 2–5 %.

In [ ]:
SYNTH_TRUTH = {
    "n_prec": 5, "n_targ": 1, "phi": 1 / 8, "n_shots": 10_000,
    "clbit_to_phys": [37, 12, 45, 20, 3],      # scrambled ON PURPOSE
    "scales": [1, 3, 5],
    "p_correct_lambda1": 0.48,                 # decays as p**lambda
    "spread_tau": 1.6,                         # off-bin leakage decay constant
    "eps0": [0.021, 0.034, 0.017, 0.048, 0.026],   # p(read 1 | true 0)
    "eps1": [0.052, 0.041, 0.063, 0.038, 0.055],   # p(read 0 | true 1)
}


def _synth_counts(true_bits_per_shot, clbit_to_phys, eps0, eps1, rng):
    """
    Apply per-qubit readout flips and emit Braket-ordered bitstrings.

    Braket returns one character per MEASURED QUBIT in ascending physical
    index order, so the character position of clbit k is the rank of
    clbit_to_phys[k] among the sorted physical qubits -- not k.
    """
    order = sorted(clbit_to_phys)
    pos_of_clbit = [order.index(q) for q in clbit_to_phys]
    n = len(clbit_to_phys)
    counts = {}
    for bits in true_bits_per_shot:
        chars = ["0"] * n
        for k in range(n):
            b = bits[k]
            flip = rng.random() < (eps1[k] if b else eps0[k])
            chars[pos_of_clbit[k]] = str(b ^ int(flip))
        s = "".join(chars)
        counts[s] = counts.get(s, 0) + 1
    return counts


def make_synthetic_results(t=SYNTH_TRUTH, seed=7, n_batches=3, drift=0.015):
    """
    n_batches x 5 payload dicts in exactly the shape checkRetrieve.py writes.

    P_correct is walked DOWN by `drift` per batch to imitate calibration decay
    over the reservation window. That is what makes the per-batch anchor table
    in section 2b a real diagnostic rather than decoration -- with identical
    batches it could never show anything.
    """
    out = []
    # The `none` arm: n_batches repeats, newest, so it is what section 2
    # selects by default.
    for b in range(n_batches):
        tb = dict(t, p_correct_lambda1=t["p_correct_lambda1"] - b * drift)
        out += _make_one_batch(tb, seed + 100 * b,
                               batch_id=f"20260807T12{10 + 15 * b:02d}00Z",
                               hour=12, minute=10 + 15 * b, ddd="none")
    # Two DDD arms, submitted earlier. XY4 genuinely helps; II (the null
    # control) barely moves -- which is what lets section 7b attribute the
    # XY4 gain to the pulses rather than to merely having added gates.
    for i, (seq, bump) in enumerate((("XY4", 0.020), ("II", 0.004))):
        tb = dict(t, p_correct_lambda1=t["p_correct_lambda1"] + bump)
        out += _make_one_batch(tb, seed + 900 + i,
                               batch_id=f"20260807T11{10 + 20 * i:02d}00Z",
                               hour=11, minute=10 + 20 * i, ddd=seq)
    return out


def _make_one_batch(t, seed, batch_id, hour, minute, ddd="none"):
    """One invocation's worth: lambda=1,3,5 plus cal0/cal1."""
    rng = np.random.default_rng(seed)
    n_prec, n_shots = t["n_prec"], t["n_shots"]
    ideal_bin = int(round(t["phi"] * 2 ** n_prec))
    dim = 2 ** n_prec
    order = sorted(t["clbit_to_phys"])

    def bits_of(m):
        return [(m >> (n_prec - 1 - k)) & 1 for k in range(n_prec)]

    base = {
        "qpu": "Cepheus-1-108Q (SYNTHETIC)",
        "circuit_type": f"QPE_{t['n_targ']}targ",
        "error_mitigation": "both", "verbatim": True, "n_stages": 1,
        "state_mode": "known", "phi": t["phi"], "ideal_bin": float(ideal_bin),
        "ideal_distribution": {str(ideal_bin): 1.0},
        "n_prec": n_prec, "n_targ": t["n_targ"],
        "n_qubits": n_prec + t["n_targ"],
        "physical_qubits": order,
        "clbit_to_phys": list(t["clbit_to_phys"]),
        "measured_qubits": order,
        "zne_scales_all": t["scales"],
        "n_shots": n_shots, "seed": 10, "target_init_seed": 99,
        "ddd_sequence": ddd,
        "ddd_report": (None if ddd == "none"
                       else {"sequence": ddd, "n_pulses": 100, "n_windows": 25}),
        "batch_id": batch_id,
        "synthetic": True,
    }

    out = []
    for scale in t["scales"]:
        # True (pre-readout) distribution: correct bin decays with lambda,
        # the lost mass leaks onto neighbours with an exponential profile.
        w = t["p_correct_lambda1"] ** scale
        leak = np.exp(-np.abs(np.arange(dim) - ideal_bin) / t["spread_tau"])
        leak[ideal_bin] = 0.0
        p = leak / leak.sum() * (1.0 - w)
        p[ideal_bin] = w
        ms = rng.choice(dim, size=n_shots, p=p)
        counts = _synth_counts((bits_of(m) for m in ms), t["clbit_to_phys"],
                               t["eps0"], t["eps1"], rng)
        out.append({**base, "job_id": f"synthetic/qpe-lambda{scale}",
                    "datetime": f"2026-08-07T{hour:02d}:{minute:02d}:{scale:02d}+00:00",
                    "role": "qpe", "label": f"QPE lambda={scale}",
                    "cal_state": None, "zne_scale": scale, "counts": counts,
                    "n_1q_gates": 210 * scale, "n_2q_gates": 96 * scale,
                    "n_gates": 306 * scale, "depth": 141 * scale,
                    "depth_2q": 62 * scale, "est_fidelity": 0.97 ** (96 * scale),
                    "est_runtime_parallel_s": 15.0 + n_shots * 2.02e-4,
                    "task_time_seconds": 18.4 + 2.1 * scale,
                    "native_quil_metadata": {
                        "gateVolume": 306 * scale, "gateDepth": 141 * scale,
                        "multiQubitGateDepth": 62 * scale,
                        "topologicalSwaps": 4 * scale,
                        "programFidelity": 0.97 ** (96 * scale),
                        "qpuRuntimeEstimation": 1000 * (2.0 + 0.4 * scale)}})

    for label, bit in (("cal0", 0), ("cal1", 1)):
        rows = [[bit] * n_prec] * n_shots
        counts = _synth_counts(rows, t["clbit_to_phys"], t["eps0"], t["eps1"], rng)
        out.append({**base, "job_id": f"synthetic/{label}",
                    "datetime": f"2026-08-07T{hour:02d}:{minute:02d}:10+00:00",
                    "role": "readout_cal", "label": f"readout calibration {label}",
                    "cal_state": label, "zne_scale": 1, "counts": counts,
                    "n_1q_gates": n_prec * bit, "n_2q_gates": 0,
                    "n_gates": n_prec * bit, "depth": 1 + bit, "depth_2q": 0,
                    "est_fidelity": 1.0, "est_runtime_parallel_s": 15.0,
                    "task_time_seconds": 16.1, "native_quil_metadata": None})
    return out


if USE_SYNTHETIC:
    print("SYNTHETIC MODE -- no hardware data is being read.")
    print(f"  ground truth: P_correct(lambda=1) = {SYNTH_TRUTH['p_correct_lambda1']}")
    print(f"                clbit_to_phys       = {SYNTH_TRUTH['clbit_to_phys']}  (scrambled)")
    print(f"                mean readout error  = "
          f"{np.mean(SYNTH_TRUTH['eps0'] + SYNTH_TRUTH['eps1']):.3f}")

## 2. Load and group into a run family

One `submit_qpe.py` invocation emits several tasks that only make sense
together: the λ series and the two calibrations share a unitary, a qubit
placement and a shot count. The directory accumulates *many* such runs side by
side, and mixing two of them silently produces a confident wrong answer — so
group first, select one family, then analyse.

Family key: `(n_prec, n_targ, state_mode, φ, zne_scales_all)`. Within a family,
the newest task wins each `(role, zne_scale, cal_state)` slot.

In [ ]:
def load_results(results_dir, use_synthetic):
    if use_synthetic:
        return make_synthetic_results()
    # Exclude this notebook's own exports: they live in the same directory and
    # would otherwise be re-read as if they were task results on the next run.
    paths = sorted(p for p in glob.glob(os.path.join(results_dir, "*.json"))
                   if not os.path.basename(p).startswith("analysis_summary"))
    if not paths:
        raise FileNotFoundError(
            f"No *.json in {results_dir}.\n"
            f"  Run:  python checkRetrieve.py job_results_qpe\n"
            f"  Or set USE_SYNTHETIC = True in section 0."
        )
    out, skipped = [], []
    for p in paths:
        with open(p) as f:
            r = json.load(f)
        r["_path"] = os.path.basename(p)
        # A result with no counts or no role cannot be placed in a family --
        # it is a partial write, or a task retrieved before it completed.
        # Dropping it loudly beats letting it become its own phantom family
        # and (being newest) get selected for analysis.
        if not r.get("counts") or not r.get("role"):
            skipped.append(r["_path"])
            continue
        out.append(r)
    if skipped:
        print(f"Skipped {len(skipped)} file(s) with no counts/role: "
              f"{', '.join(skipped[:4])}{' ...' if len(skipped) > 4 else ''}")
    return out


def family_key(r):
    """
    What makes two tasks part of the same experiment.

    `ddd_sequence` is IN the key: DDD-on and DDD-off are DIFFERENT CIRCUITS,
    and pooling them would average a decoupled run together with an
    undecoupled one and report the mean as if it were one condition. It is
    emphatically not a batch dimension -- batches are repeats of an identical
    config, DDD arms are not.

    Batch is NOT in the key. Repeats of one config belong to the same family
    and are separated afterwards by assign_batches(), so they can be pooled
    for the headline number or compared against each other for drift.
    """
    return (r.get("n_prec"), r.get("n_targ"), r.get("state_mode"),
            r.get("phi"), tuple(r.get("zne_scales_all") or [r.get("zne_scale", 1)]),
            r.get("ddd_sequence") or "none")


def assign_batches(records, gap_s):
    """
    Split one family into batches -- one `submit_qpe.py` invocation each.

    Prefers the `batch_id` stamped at submission time. Results written before
    that field existed fall back to clustering on submission time: the tasks
    of a single invocation land seconds apart, while separate invocations are
    minutes apart. Returns (ordered {batch -> records}, provenance string).
    """
    if records and all(r.get("batch_id") for r in records):
        out = {}
        for r in sorted(records, key=_when):
            out.setdefault(r["batch_id"], []).append(r)
        return out, "logged batch_id"

    out, cur, prev, idx = {}, [], None, 1
    for r in sorted(records, key=_when):
        t = _when(r)
        if prev is not None and (t - prev).total_seconds() > gap_s:
            out[f"inferred-{idx}"] = cur
            cur, idx = [], idx + 1
        cur.append(r)
        prev = t
    if cur:
        out[f"inferred-{idx}"] = cur
    return out, f"inferred from timestamps (gap > {gap_s}s)"


def slot_tasks(records):
    """(qpe_by_scale, cal_by_state) for one batch; later wins a duplicate."""
    qpe, cal = {}, {}
    for r in sorted(records, key=_when):
        if r.get("role") == "qpe":
            qpe[int(r["zne_scale"])] = r
        elif r.get("role") == "readout_cal":
            cal[r["cal_state"]] = r
    return qpe, cal


def pool_counts(records):
    """
    Merge several batches' versions of the SAME slot into one record.

    Counts are summed, which is exactly right for independent repeats of an
    identical circuit -- 3 x 10,000 shots is one 30,000-shot estimate. Every
    other field is taken from the first record because the batches agree on
    all of them by construction (same family key, same placement).

    What this DOES hide is drift: summing assumes the batches are identically
    distributed, which on a superconducting device over an hour they are not.
    That is what section 2b measures separately, and why pooling is offered
    alongside per-batch rather than instead of it.
    """
    merged = dict(records[0])
    counts = {}
    for r in records:
        for k, v in r["counts"].items():
            counts[k] = counts.get(k, 0) + v
    merged["counts"] = counts
    merged["n_shots"] = sum(counts.values())
    merged["_pooled_from"] = [r.get("batch_id") or r["job_id"] for r in records]
    return merged


def _when(r):
    """
    Submission time, ALWAYS timezone-aware.

    checkRetrieve.py writes whatever isoformat the submitting script produced,
    and older records carry no offset. Mixing the two blows up on the first
    comparison with "can't compare offset-naive and offset-aware datetimes",
    which is a sorting bug, not a data problem -- so normalise here rather
    than at every call site. Naive stamps are read as UTC, which is what the
    submit scripts record.
    """
    try:
        t = datetime.fromisoformat(r["datetime"])
    except Exception:
        return datetime.min.replace(tzinfo=timezone.utc)
    return t if t.tzinfo is not None else t.replace(tzinfo=timezone.utc)


results = load_results(RESULTS_DIR, USE_SYNTHETIC)
print(f"Loaded {len(results)} task result(s).\n")

families = {}
for r in results:
    families.setdefault(family_key(r), []).append(r)

print(f"{len(families)} run famil{'y' if len(families) == 1 else 'ies'} found:")
for k, v in sorted(families.items(), key=lambda kv: max(_when(r) for r in kv[1])):
    n_prec, n_targ, mode, phi, scales, ddd = k
    nb = len(assign_batches(v, BATCH_GAP_S)[0])
    newest = max(_when(r) for r in v).isoformat(timespec="seconds")
    print(f"  n_prec={n_prec} n_targ={n_targ} state={mode} phi={phi} "
          f"scales={list(scales)} ddd={ddd} | {len(v)} task(s) in {nb} batch(es) "
          f"| newest {newest}")

# -- Select -----------------------------------------------------------------
if SELECT_RUN is None:
    key = max(families, key=lambda k: max(_when(r) for r in families[k]))
else:
    want = tuple(SELECT_RUN)
    matches = [k for k in families if k[:len(want)] == want]
    if not matches:
        raise KeyError(f"SELECT_RUN={SELECT_RUN} matches no family.")
    key = max(matches, key=lambda k: max(_when(r) for r in families[k]))

fam = families[key]
N_PREC, N_TARG, STATE_MODE, PHI, SCALES_LOGGED, DDD_SEQ = key
print(f"\n--> selected: n_prec={N_PREC} n_targ={N_TARG} state={STATE_MODE} "
      f"phi={PHI} ddd={DDD_SEQ}")

# -- Split into batches -----------------------------------------------------
BATCHES, BATCH_PROV = assign_batches(fam, BATCH_GAP_S)
print(f"    batches           : {len(BATCHES)}  ({BATCH_PROV})")
for bid, recs in BATCHES.items():
    sc = sorted(int(r["zne_scale"]) for r in recs if r.get("role") == "qpe")
    cs = sorted(r["cal_state"] for r in recs if r.get("role") == "readout_cal")
    print(f"      {bid:<20} {len(recs)} task(s)  lambda={sc}  cal={cs}")

# -- Choose what the main analysis runs on ----------------------------------
if BATCH_MODE == "pooled":
    slots = [slot_tasks(recs) for recs in BATCHES.values()]
    qpe_by_scale, cal_by_state = {}, {}
    for s in sorted({s for q, _ in slots for s in q}):
        group = [q[s] for q, _ in slots if s in q]
        qpe_by_scale[s] = pool_counts(group)
    for cs in sorted({c for _, cal in slots for c in cal}):
        group = [cal[cs] for _, cal in slots if cs in cal]
        cal_by_state[cs] = pool_counts(group)
    print(f"\n    analysing         : POOLED across {len(BATCHES)} batch(es)")
else:
    if BATCH_MODE not in BATCHES:
        raise KeyError(f"BATCH_MODE={BATCH_MODE!r} is not one of "
                       f"{list(BATCHES)} (or 'pooled').")
    qpe_by_scale, cal_by_state = slot_tasks(BATCHES[BATCH_MODE])
    print(f"\n    analysing         : batch {BATCH_MODE} only")

SCALES  = sorted(qpe_by_scale)
HAS_ZNE = len(SCALES) > 1
HAS_REM = {"cal0", "cal1"} <= set(cal_by_state)

print(f"    QPE tasks    : lambda = {SCALES}")
print(f"    calibrations : {sorted(cal_by_state) or 'none'}")
print(f"    ZNE possible : {HAS_ZNE}      REM possible : {HAS_REM}")

if not HAS_ZNE:
    print("    NOTE: one lambda only -- ZNE sections will be skipped.")
if not HAS_REM:
    print("    NOTE: cal0/cal1 missing -- REM sections will be skipped.")

# -- Consistency guards -----------------------------------------------------
shots = {r["job_id"]: sum(r["counts"].values()) for r in fam}
print(f"\n    shots per task: {sorted(set(shots.values()))}")
if not USE_SYNTHETIC:
    verb = {bool(r.get("verbatim")) for r in fam}
    if HAS_ZNE and verb != {True}:
        print("\n!!  WARNING: verbatim is not True on every task in this family.")
        print("    Rigetti's Quilc cancels the C-dag-C folds without it, so every")
        print("    lambda may have executed as lambda=1 and the extrapolation")
        print("    below would be meaningless. Check `verbatim` in the job log.")

## 3. The clbit → physical map

The one place this analysis can go wrong quietly.

Our QPE writes precision qubit *k* into classical bit *k*, with prec[0] the MSB,
so `m = Σ c[k]·2^(n_prec−1−k)`. But Braket does not return `c[k]`. It returns
**one character per measured qubit, ordered by ascending physical index**, and
after SABRE routing that order has no relationship to clbit order at all.
Decoding needs the explicit `clbit → physical` list.

Resolution order:

1. **`clbit_to_phys` in the record** — logged at submission. Authoritative.
2. **Re-derive it** — rebuild and re-transpile the circuit from the logged
   unitary and seeds, then cross-check that its qubit *set* matches the logged
   `physical_qubits`. Covers results written before that field existed.
3. **Assume sorted order** — last resort, and *loudly* flagged. `physical_qubits`
   is stored sorted, which throws the ordering away; using it asserts "clbit *k*
   is the *k*-th smallest physical qubit," which routing does not guarantee.

In [ ]:
def resolve_clbit_to_phys(rec, n_meas):
    """Return (mapping, provenance). See the three tiers above."""
    ctp = rec.get("clbit_to_phys")
    if ctp:
        return [int(q) for q in ctp], "logged"

    # Tier 2: rebuild the circuit and re-run the same deterministic routing.
    # Both routing entry points are tried, because which one the submission
    # used depends on the script version -- place_and_route (chiplet-aware,
    # current) picks DIFFERENT physical qubits than a bare transpile, so
    # guessing wrong here would produce a wrong map that still looks sane.
    # The set cross-check against `physical_qubits` is what makes it safe.
    logged = sorted(int(q) for q in rec.get("physical_qubits", []))
    try:
        u = rec["unitary"]
        U = np.array(u["real"], dtype=float) + 1j * np.array(u["imag"], dtype=float)
        caps, caps_real = load_device_caps()
        if not caps_real:
            raise RuntimeError("device_caps.json absent -- routing would differ")
        qc = build_qpe_circuit(
            rec["n_prec"], rec["n_targ"], U,
            target_init_seed=rec.get("target_init_seed"),
            eigenstate=(rec.get("state_mode") == "known"),
        )
        routers = []
        if place_and_route is not None:
            routers.append(("place_and_route", lambda c: place_and_route(
                c, caps, hub_qubits=list(range(rec["n_targ"])), verbose=False)))
        routers.append(("transpile_for_rigetti",
                        lambda c: transpile_for_rigetti(c, caps)))
        for name, route in routers:
            derived = measured_physical_qubits(route(qc))
            if logged and sorted(int(q) for q in derived) == logged:
                return [int(q) for q in derived], f"re-derived ({name})"
        raise RuntimeError(
            f"no router reproduced the logged qubit set {logged}; "
            "qiskit version, transpiler seed or placement policy has changed"
        )
    except Exception:
        pass

    phys = [int(q) for q in rec.get("physical_qubits", [])][:n_meas]
    if len(phys) < n_meas:
        raise RuntimeError("cannot establish a clbit -> physical map for this record")
    return phys, "ASSUMED-SORTED"


ref = qpe_by_scale[SCALES[0]]
MEASURED_QUBITS = [int(q) for q in ref["measured_qubits"]]
N_MEAS = len(MEASURED_QUBITS)
CLBIT_TO_PHYS, PROVENANCE = resolve_clbit_to_phys(ref, N_MEAS)

print(f"measured qubits (Braket char order) : {MEASURED_QUBITS}")
print(f"clbit -> physical                   : {CLBIT_TO_PHYS}")
print(f"provenance                          : {PROVENANCE}")

if PROVENANCE == "ASSUMED-SORTED":
    print("\n!!  The clbit -> physical order could not be established.")
    print("    Assuming clbit k == k-th smallest physical qubit. If routing")
    print("    did not happen to assign them that way, EVERY number below is")
    print("    wrong in a way that still looks plausible. Fix by re-running")
    print("    with the current submit_qpe.py, which logs `clbit_to_phys`.")

# Every task in the family must share the map, or the cals correct the wrong bits.
for r in fam:
    mq = [int(q) for q in r["measured_qubits"]]
    if mq != MEASURED_QUBITS:
        raise ValueError(
            f"task {r['job_id']} measured {mq}, expected {MEASURED_QUBITS}. "
            "These tasks are not from one placement and cannot be combined."
        )

CONV = dict(convention="braket", clbit_to_phys=CLBIT_TO_PHYS,
            measured_qubits=MEASURED_QUBITS)
print("\nAll tasks agree on the measured-qubit order.")

### 3a. Does the mapping actually matter?

Decode the λ=1 counts twice — once with the resolved map, once under the naive
"sorted" assumption — and compare. If the two disagree, the sorted shortcut
would have silently reported a different distribution.

In [ ]:
IDEAL_BIN = (int(round(ref["ideal_bin"])) if ref.get("ideal_bin") is not None
             else int(max((ref.get("ideal_distribution") or {"0": 1.0}).items(),
                          key=lambda kv: kv[1])[0]))

_raw = qpe_by_scale[SCALES[0]]["counts"]
p_good = ma.to_probabilities(ma.counts_to_m(_raw, N_PREC, **CONV), N_PREC)
p_naive = ma.to_probabilities(
    ma.counts_to_m(_raw, N_PREC, convention="braket",
                   clbit_to_phys=sorted(CLBIT_TO_PHYS),
                   measured_qubits=MEASURED_QUBITS), N_PREC)

print(f"ideal bin (phi * 2^n_prec) : {IDEAL_BIN}")
print(f"resolved map  -> argmax {int(np.argmax(p_good)):>3}   "
      f"P_correct {p_good[IDEAL_BIN]:.4f}")
print(f"sorted assump -> argmax {int(np.argmax(p_naive)):>3}   "
      f"P_correct {p_naive[IDEAL_BIN]:.4f}")

if np.allclose(p_good, p_naive):
    print("\nIdentical here -- routing happened to assign clbits in ascending")
    print("physical order. That is luck, not a guarantee; keep the explicit map.")
else:
    print(f"\nDIFFERENT. The sorted shortcut misplaces "
          f"{0.5 * np.abs(p_good - p_naive).sum():.1%} of the distribution.")
    print("This is exactly the failure the explicit map exists to prevent.")

if int(np.argmax(p_good)) != IDEAL_BIN:
    print(f"\nNOTE: the decoded peak ({int(np.argmax(p_good))}) is not the ideal")
    print(f"      bin ({IDEAL_BIN}). Either the device is noisier than the signal,")
    print("      or the bit order is still wrong. Check against a low-depth run.")

## 4. Distributions

Decode every λ into a probability vector over the QPE integer *m*, and build
the noiseless reference from the `ideal_distribution` logged at submission.

In [ ]:
counts_by_scale = {s: qpe_by_scale[s]["counts"] for s in SCALES}
p_by_scale = {s: ma.to_probabilities(ma.counts_to_m(c, N_PREC, **CONV), N_PREC)
              for s, c in counts_by_scale.items()}

DIM = 2 ** N_PREC
ideal = np.zeros(DIM)
for m, p in (ref.get("ideal_distribution") or {}).items():
    if int(m) < DIM:
        ideal[int(m)] = p
if ideal.sum() == 0:
    ideal[IDEAL_BIN] = 1.0
    print("NOTE: no ideal_distribution logged -- using a delta at the ideal bin.")

print(f"n_prec = {N_PREC}  ->  {DIM} bins;  ideal bin = {IDEAL_BIN}"
      f"  (phi = {PHI})\n")
print(f"{'lambda':>7} {'shots':>8} {'argmax':>7} {'P_correct':>10} "
      f"{'TVD':>8} {'phase err':>10}")
for s in SCALES:
    p = p_by_scale[s]
    pe = ma.phase_error(p, PHI, N_PREC) if PHI is not None else float("nan")
    print(f"{s:>7} {sum(counts_by_scale[s].values()):>8,} "
          f"{int(np.argmax(p)):>7} {ma.p_correct(p, IDEAL_BIN):>10.4f} "
          f"{ma.tvd(p, ideal):>8.4f} {pe:>10.4f}")

## 4b. Batch-to-batch drift

The reason to run three batches rather than one long batch. Each is a repeat
of an identical config minutes apart, so the **spread of the λ=1 anchor across
batches is a direct measurement of calibration drift** over the window.

This is the term the bootstrap CIs elsewhere in this notebook do *not* contain.
They resample counts, so they capture shot noise only; two tasks twenty minutes
apart on a superconducting device are not identically distributed. If the
across-batch spread here is comfortably larger than the within-batch shot
noise, the bootstrap intervals understate your real uncertainty and this is
the number to quote alongside them.

Pooling (`BATCH_MODE = "pooled"`) sums the batches and therefore assumes the
drift is negligible. Read this table before trusting that assumption.

In [ ]:
def batch_anchor(recs):
    """Raw and REM-corrected P_correct for one batch's lambda=1 task."""
    qpe, cal = slot_tasks(recs)
    base = min(qpe) if qpe else None
    if base is None:
        return None
    p = ma.to_probabilities(ma.counts_to_m(qpe[base]["counts"], N_PREC, **CONV),
                            N_PREC)
    row = {"scale": base, "shots": sum(qpe[base]["counts"].values()),
           "raw": ma.p_correct(p, IDEAL_BIN), "rem": None}
    if {"cal0", "cal1"} <= set(cal):
        a = ma.tensored_confusion(
            ma.confusion_matrices(cal["cal0"]["counts"], cal["cal1"]["counts"],
                                  N_MEAS, **CONV)[:N_PREC])
        row["rem"] = ma.p_correct(ma.rem_correct(p, a, method=REM_METHOD),
                                  IDEAL_BIN)
    return row


anchors = {b: batch_anchor(r) for b, r in BATCHES.items()}
anchors = {b: a for b, a in anchors.items() if a}

print(f"{'batch':<22} {'shots':>8} {'P_raw':>9} {'shot sd':>9} {'P_rem':>9}")
for b, a in anchors.items():
    sd = np.sqrt(max(a["raw"] * (1 - a["raw"]), 1e-12) / a["shots"])
    rem = "     --" if a["rem"] is None else f"{a['rem']:>9.4f}"
    print(f"{b:<22} {a['shots']:>8,} {a['raw']:>9.4f} {sd:>9.4f} {rem}")

if len(anchors) < 2:
    print("\nOne batch only -- no drift estimate available. Repeat the same "
          "config in a later invocation to get one.")
else:
    vals = np.array([a["raw"] for a in anchors.values()])
    sds  = np.array([np.sqrt(max(v * (1 - v), 1e-12) / a["shots"])
                     for v, a in zip(vals, anchors.values())])
    spread, shot = float(vals.max() - vals.min()), float(sds.mean())
    print(f"\n  across-batch spread (max-min) : {spread:.4f}")
    print(f"  mean within-batch shot sd     : {shot:.4f}")
    print(f"  ratio                         : {spread / shot:.1f}x")
    print(f"  pooled mean +/- sd            : {vals.mean():.4f} +/- "
          f"{vals.std(ddof=1) if len(vals) > 1 else 0.0:.4f}")
    if spread > 3 * shot:
        print("\n  DRIFT DOMINATES SHOT NOISE. The bootstrap CIs in sections 7")
        print("  and 8 are shot-noise only and therefore UNDERSTATE the real")
        print("  uncertainty. Quote the across-batch sd above alongside them,")
        print("  and treat any mitigation delta smaller than it as unresolved.")
    else:
        print("\n  Drift is within shot noise -- pooling the batches is safe and")
        print("  the bootstrap CIs can be read as the total uncertainty.")

## 5. REM — inverting the readout channel

`cal0` (all-|0⟩) and `cal1` (all-|1⟩) give a 2×2 confusion matrix per qubit.
Tensoring them gives the full `A` with `p_noisy = A · p_true`, which we invert.

Two things to read off:

- **Condition number** — how much shot noise `A⁻¹` amplifies. This is the
  variance half of REM's bias/variance trade. Near 1 means the correction is
  nearly free; log it every run.
- **Method** — `simplex` solves a constrained least squares keeping the result
  a probability distribution. `naive` (`A⁻¹p`, clip, renormalise) is what most
  papers report and is *biased*: clipping pushes weight into surviving bins.
  Both are computed; `simplex` is used downstream.

In [ ]:
if HAS_REM:
    cal0_counts = cal_by_state["cal0"]["counts"]
    cal1_counts = cal_by_state["cal1"]["counts"]

    mats = ma.confusion_matrices(cal0_counts, cal1_counts, N_MEAS, **CONV)
    A = ma.tensored_confusion(mats[:N_PREC])
    COND = ma.condition_number(A)

    print("Per-qubit readout confusion (clbit order, prec[0] = MSB):\n")
    print(f"{'clbit':>6} {'phys':>5} {'p(1|0)':>9} {'p(0|1)':>9} {'fidelity':>9}")
    for k, a in enumerate(mats[:N_PREC]):
        p10, p01 = a[1, 0], a[0, 1]
        print(f"{k:>6} {CLBIT_TO_PHYS[k]:>5} {p10:>9.4f} {p01:>9.4f} "
              f"{1 - 0.5 * (p10 + p01):>9.4f}")

    print(f"\nA is {A.shape[0]}x{A.shape[1]};  condition number = {COND:.3f}")
    if COND > 10:
        print("  WARNING: cond > 10 -- the inversion is amplifying shot noise")
        print("  substantially. Widen the bootstrap interpretation accordingly.")

    p_rem_by_scale = {s: ma.rem_correct(p, A, method=REM_METHOD)
                      for s, p in p_by_scale.items()}
    p_rem_naive = ma.rem_correct(p_by_scale[SCALES[0]], A, method="naive")

    print(f"\n{'lambda':>7} {'P_raw':>9} {'P_rem':>9} {'delta':>9}")
    for s in SCALES:
        a_, b_ = ma.p_correct(p_by_scale[s], IDEAL_BIN), ma.p_correct(p_rem_by_scale[s], IDEAL_BIN)
        print(f"{s:>7} {a_:>9.4f} {b_:>9.4f} {b_ - a_:>+9.4f}")

    print(f"\nmethod comparison at lambda={SCALES[0]}:  "
          f"simplex {ma.p_correct(p_rem_by_scale[SCALES[0]], IDEAL_BIN):.4f}   "
          f"naive {ma.p_correct(p_rem_naive, IDEAL_BIN):.4f}")
else:
    p_rem_by_scale, A, COND = None, None, None
    print("REM skipped -- no cal0/cal1 in this family.")

## 6. ZNE — extrapolating to zero noise

`P_correct(λ)` is fitted under three models and evaluated at λ=0. Reporting all
three is deliberate: **the spread between models is part of the uncertainty**,
and Richardson disagreeing wildly with linear is itself the finding — it says
the noise is outside the regime where extrapolation is valid.

The physicality check is the one that matters. P_correct is a probability; an
extrapolation landing outside [0,1] is not a slightly-off estimate, it is proof
the fit is invalid. `mitig_analysis.extrapolate` flags that rather than
reporting an impossible number as a result.

In [ ]:
def show_fit(title, fit):
    print(f"{title}")
    for m in ZNE_MODELS:
        v = fit.get(m)
        if v is None:
            print(f"    {m:<14} failed: {fit.get(m + '_error', 'no fit')}")
        else:
            flag = "  <-- OUTSIDE [0,1], INVALID" if not (0 <= v <= 1) else ""
            print(f"    {m:<14} {v:>8.4f}{flag}")
    if "model_spread" in fit:
        print(f"    {'model spread':<14} {fit['model_spread']:>8.4f}")
    if not fit.get("physical", True):
        print(f"    VERDICT: unphysical under {fit['unphysical']} -- "
              f"do not report as a mitigated value.")
    print()


zne_fit = zne_rem_fit = None
if HAS_ZNE:
    vals_raw = [ma.p_correct(p_by_scale[s], IDEAL_BIN) for s in SCALES]
    print(f"P_correct by lambda (raw)  : "
          + "  ".join(f"L{s}={v:.4f}" for s, v in zip(SCALES, vals_raw)) + "\n")
    zne_fit = ma.extrapolate(SCALES, vals_raw, models=ZNE_MODELS)
    show_fit("ZNE (no readout correction), extrapolated to lambda=0:", zne_fit)

    if HAS_REM:
        vals_rem = [ma.p_correct(p_rem_by_scale[s], IDEAL_BIN) for s in SCALES]
        print(f"P_correct by lambda (REM)  : "
              + "  ".join(f"L{s}={v:.4f}" for s, v in zip(SCALES, vals_rem)) + "\n")
        zne_rem_fit = ma.extrapolate(SCALES, vals_rem, models=ZNE_MODELS)
        show_fit("ZNE+REM, extrapolated to lambda=0:", zne_rem_fit)
        print("Note: REM lifts every point on the curve, and the extrapolation")
        print("multiplies that shift by its lever arm -- which is why ZNE+REM")
        print("goes unphysical sooner than either technique alone.")
else:
    print("ZNE skipped -- only one lambda in this family.")

## 7. Bootstrap confidence intervals

Three different resampling structures, because the comparisons differ:

- **REM** is a *deterministic map on the same counts*, not a second
  experiment — so REM-on and REM-off must be resampled **jointly** (paired).
  Combining two independent error bars in quadrature would badly overstate the
  uncertainty on Δ. The calibration counts are resampled too, so
  confusion-matrix uncertainty propagates in.
- **ZNE** is bootstrapped as a **whole fit** — resample every λ, refit,
  extrapolate — because the lever arm inflates variance far past what any
  single point's standard error suggests.
- **ZNE+REM** shares one calibration draw across all λ, matching how a single
  calibration corrects the whole series.

In [ ]:
def bootstrap_zne_rem(counts_by_scale, cal0, cal1, n_prec, ideal_bin, n_meas,
                      n_boot, seed, models, method, **conv):
    """
    Bootstrap the REM-then-extrapolate chain.

    One calibration resample per draw, SHARED across every lambda -- because on
    hardware one pair of calibration tasks corrects the whole lambda series.
    Drawing a fresh calibration per lambda would pretend the correction is
    independent at each point and understate the correlated component.
    """
    rng = np.random.default_rng(seed)
    scales = sorted(counts_by_scale)
    per_model = {m: [] for m in models}
    for _ in range(n_boot):
        a = ma.tensored_confusion(
            ma.confusion_matrices(ma.resample_counts(cal0, rng),
                                  ma.resample_counts(cal1, rng),
                                  n_meas, **conv)[:n_prec])
        vals = []
        for s in scales:
            p = ma.to_probabilities(
                ma.counts_to_m(ma.resample_counts(counts_by_scale[s], rng),
                               n_prec, **conv), n_prec)
            vals.append(ma.p_correct(ma.rem_correct(p, a, method=method), ideal_bin))
        fit = ma.extrapolate(scales, vals, models=models)
        for m in models:
            per_model[m].append(fit.get(m))
    return {m: ma.bootstrap_ci(v) for m, v in per_model.items()}


def fmt_ci(ci):
    if ci is None or ci.get("mean") is None:
        return "     --"
    return f"{ci['mean']:.4f} [{ci['lo']:+.4f}, {ci['hi']:+.4f}]"


rem_boot = zne_boot = zne_rem_boot = None

if HAS_REM:
    rem_boot = ma.bootstrap_rem_paired(
        counts_by_scale[SCALES[0]], cal0_counts, cal1_counts,
        N_PREC, IDEAL_BIN, N_MEAS, n_boot=N_BOOT, method=REM_METHOD,
        seed=SEED, **CONV)
    d = rem_boot["delta"]
    sig = d["lo"] > 0 or d["hi"] < 0
    print("REM, paired bootstrap at lambda=1")
    print(f"    raw            {fmt_ci(rem_boot['raw'])}")
    print(f"    REM            {fmt_ci(rem_boot['rem'])}")
    print(f"    delta          {fmt_ci(d)}   "
          f"{'SIGNIFICANT' if sig else 'not significant (CI spans 0)'}")
    print(f"    cond number    {fmt_ci(rem_boot['condition_number'])}\n")

if HAS_ZNE:
    zne_boot = ma.bootstrap_zne(counts_by_scale, N_PREC, IDEAL_BIN,
                                n_boot=N_BOOT, seed=SEED, models=ZNE_MODELS,
                                **CONV)
    print("ZNE, bootstrapped over the whole fit")
    for m in ZNE_MODELS:
        print(f"    {m:<14} {fmt_ci(zne_boot['ci'][m])}")
    print()

    if HAS_REM:
        zne_rem_boot = bootstrap_zne_rem(
            counts_by_scale, cal0_counts, cal1_counts, N_PREC, IDEAL_BIN,
            N_MEAS, N_BOOT, SEED, ZNE_MODELS, REM_METHOD, **CONV)
        print("ZNE+REM, bootstrapped over the whole chain")
        for m in ZNE_MODELS:
            print(f"    {m:<14} {fmt_ci(zne_rem_boot[m])}")
        print()

print("Shot noise only. Two tasks minutes apart on a superconducting device")
print("are not identically distributed -- add a drift term estimated from a")
print("repeated lambda=1 anchor before quoting these as total uncertainty.")

## 7b. DDD — comparing decoupling arms

DDD costs no extra tasks, so it does not appear as a row in the mitigation
table above: it is a different **circuit**, submitted as its own run, and lives
in its own run family. This section finds families that match the selected one
in every respect *except* `ddd_sequence` and compares their λ=1 anchors.

The comparison is **unpaired** — different circuits, submitted at different
times — so the interval is genuinely wider than the paired REM one, and it
carries the drift caveat from §4b on top.

Two things to read carefully:

- **`II` is the null control.** It inserts identity gates with the same count
  and placement as a real sequence. If `II` helps as much as `XY4`, the effect
  is not decoupling. A real sequence must beat `II`, not merely beat `none`.
- **`II` only means anything inside a verbatim box**, and even then it tests
  "did adding instructions help", not "did adding *pulses* help" — a compiler
  is entitled to delete identity gates outright.

In [ ]:
def pooled_lambda1(records):
    """Pooled lambda=1 counts + measured-qubit order for a whole family."""
    batches, _ = assign_batches(records, BATCH_GAP_S)
    per = [slot_tasks(r)[0] for r in batches.values()]
    base = min({s for q in per for s in q}, default=None)
    if base is None:
        return None
    merged = pool_counts([q[base] for q in per if base in q])
    return merged


siblings = {k[5]: v for k, v in families.items() if k[:5] == key[:5]}
print(f"DDD arms present for this config: {sorted(siblings)}\n")

if len(siblings) < 2:
    print("Only one DDD arm -- nothing to compare. Submit the same config with")
    print("a different --ddd value to populate this section, e.g.:")
    print(f"    python submit_qpe.py --ddd XY4   # then --ddd II for the control")
    ddd_deltas = None
else:
    baseline_seq = "none" if "none" in siblings else sorted(siblings)[0]
    base_rec = pooled_lambda1(siblings[baseline_seq])
    base_mq  = [int(q) for q in base_rec["measured_qubits"]]

    print(f"{'arm':<10} {'shots':>8} {'P_correct':>10} "
          f"{'delta vs ' + baseline_seq:>18} {'95% CI':>22}  verdict")
    ddd_deltas = {}
    for seq in sorted(siblings):
        rec = pooled_lambda1(siblings[seq])
        mq  = [int(q) for q in rec["measured_qubits"]]
        if mq != base_mq:
            print(f"{seq:<10} SKIPPED -- measured qubits {mq} != {base_mq}; "
                  f"different placement, not comparable.")
            continue
        p = ma.to_probabilities(ma.counts_to_m(rec["counts"], N_PREC, **CONV),
                                N_PREC)
        pc = ma.p_correct(p, IDEAL_BIN)
        if seq == baseline_seq:
            print(f"{seq:<10} {sum(rec['counts'].values()):>8,} {pc:>10.4f} "
                  f"{'(baseline)':>18} {'':>22}")
            continue
        bs = ma.bootstrap_delta_unpaired(
            base_rec["counts"], rec["counts"], N_PREC, IDEAL_BIN,
            n_boot=N_BOOT, seed=SEED, **CONV)
        d = bs["delta"]
        verdict = ("SIGNIFICANT" if d["significant"] else "n.s.")
        if d["significant"] and d["hi"] < 0:
            verdict += " (HURTS)"
        ddd_deltas[seq] = bs
        print(f"{seq:<10} {sum(rec['counts'].values()):>8,} {pc:>10.4f} "
              f"{d['mean']:>+18.4f} [{d['lo']:+.4f}, {d['hi']:+.4f}]  {verdict}")

    if "II" in ddd_deltas:
        null_d = ddd_deltas["II"]["delta"]["mean"]
        print(f"\n  Null control II moved P_correct by {null_d:+.4f}.")
        for seq, bs in ddd_deltas.items():
            if seq == "II":
                continue
            over = bs["delta"]["mean"] - null_d
            print(f"  {seq} beats the null control by {over:+.4f} "
                  f"-- {'the gain is the pulses' if over > 0 else 'NOT attributable to decoupling'}.")
    elif ddd_deltas:
        print("\n  No `II` arm was run, so a positive result here cannot be")
        print("  separated from 'adding any gates helped'. Submit --ddd II to")
        print("  close that gap; it costs the same as any other arm.")

## 8. Results table

Every cell in one place. `ZNE` rows carry a P_correct only: extrapolation
produces a single expectation value, not a distribution, so TVD and phase error
are undefined there — reporting them would be inventing a distribution that was
never measured.

In [ ]:
ZNE_HEADLINE = "linear"      # the model quoted in the summary row

rows = []
rows.append({
    "condition": "raw (lambda=1)",
    "p_correct": ma.p_correct(p_by_scale[SCALES[0]], IDEAL_BIN),
    "tvd": ma.tvd(p_by_scale[SCALES[0]], ideal),
    "phase_err": ma.phase_error(p_by_scale[SCALES[0]], PHI, N_PREC) if PHI else None,
    "ci": rem_boot["raw"] if rem_boot else None,
    "valid": True,
})
if HAS_REM:
    rows.append({
        "condition": f"REM ({REM_METHOD})",
        "p_correct": ma.p_correct(p_rem_by_scale[SCALES[0]], IDEAL_BIN),
        "tvd": ma.tvd(p_rem_by_scale[SCALES[0]], ideal),
        "phase_err": ma.phase_error(p_rem_by_scale[SCALES[0]], PHI, N_PREC) if PHI else None,
        "ci": rem_boot["rem"], "valid": True,
    })
if HAS_ZNE:
    v = zne_fit.get(ZNE_HEADLINE)
    rows.append({
        "condition": f"ZNE ({ZNE_HEADLINE})", "p_correct": v,
        "tvd": None, "phase_err": None,
        "ci": zne_boot["ci"][ZNE_HEADLINE],
        "valid": v is not None and 0.0 <= v <= 1.0,
    })
if HAS_ZNE and HAS_REM:
    v = zne_rem_fit.get(ZNE_HEADLINE)
    rows.append({
        "condition": f"ZNE+REM ({ZNE_HEADLINE})", "p_correct": v,
        "tvd": None, "phase_err": None,
        "ci": zne_rem_boot[ZNE_HEADLINE],
        "valid": v is not None and 0.0 <= v <= 1.0,
    })

hdr = f"{'condition':<22} {'P_correct':>10} {'95% CI':>22} {'TVD':>8} {'phase err':>10}  status"
print(hdr)
print("-" * len(hdr))
for r in rows:
    pc = "  --" if r["p_correct"] is None else f"{r['p_correct']:.4f}"
    ci = r["ci"]
    ci_s = "--" if not ci or ci.get("lo") is None else f"[{ci['lo']:.4f}, {ci['hi']:.4f}]"
    tv = "  --" if r["tvd"] is None else f"{r['tvd']:.4f}"
    pe = "  --" if r["phase_err"] is None else f"{r['phase_err']:.4f}"
    status = "ok" if r["valid"] else "UNPHYSICAL -- do not report"
    print(f"{r['condition']:<22} {pc:>10} {ci_s:>22} {tv:>8} {pe:>10}  {status}")
print("-" * len(hdr))
print(f"baseline P_correct = {rows[0]['p_correct']:.4f};  ideal = 1.0000 "
      f"(bin {IDEAL_BIN}, phi = {PHI})")

## 9. Figures

In [ ]:
# -- Figure 1: distribution at lambda=1 ------------------------------------
# Bars, not lines: m is a set of discrete labelled outcomes, and the question
# is "how much mass sits in each" -- magnitude by category.
K = 12
weight = ideal + p_by_scale[SCALES[0]] + (p_rem_by_scale[SCALES[0]] if HAS_REM else 0)
bins = sorted(np.argsort(weight)[::-1][:K])

series = [("Ideal (noiseless)", ideal, C["ideal"]),
          ("Raw", p_by_scale[SCALES[0]], C["raw"])]
if HAS_REM:
    series.append(("REM", p_rem_by_scale[SCALES[0]], C["rem"]))

x = np.arange(len(bins))
w = 0.8 / len(series)
fig, ax = plt.subplots(figsize=(10, 4.2))
for i, (label, p, colour) in enumerate(series):
    ax.bar(x + i * w - 0.4 + w / 2, [p[b] for b in bins], w * 0.88,
           label=label, color=colour, linewidth=0)

ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in bins])
ax.set_xlabel("QPE integer  m   (phase = m / 2^n_prec)")
ax.set_ylabel("probability")
ax.set_title(f"Precision-register distribution at $\\lambda$=1  "
             f"(n_prec={N_PREC}, $\\phi$={PHI}, ideal bin {IDEAL_BIN})")
ax.grid(axis="x", visible=False)
ax.legend(ncol=len(series), loc="upper right")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.2f}"))
fig.savefig(os.path.join(PLOTS_DIR, "qpe_distribution_lambda1.png"))
plt.show()

# Table view: three of these hues sit under 3:1 contrast on the surface, so
# the figure never carries identity by colour alone.
print(f"\n{'m':>5} " + " ".join(f"{lbl:>18}" for lbl, _, _ in series))
for b in bins:
    mark = "  <-- ideal" if b == IDEAL_BIN else ""
    print(f"{b:>5} " + " ".join(f"{p[b]:>18.4f}" for _, p, _ in series) + mark)

In [ ]:
# -- Figure 2: P_correct vs noise scale, with the extrapolations -----------
if HAS_ZNE:
    fig, ax = plt.subplots(figsize=(8, 4.6))
    grid = np.linspace(0, max(SCALES) * 1.05, 200)

    curves = [("Raw", [ma.p_correct(p_by_scale[s], IDEAL_BIN) for s in SCALES],
               zne_fit, C["raw"], (10, -16))]
    if HAS_REM:
        curves.append(("REM", [ma.p_correct(p_rem_by_scale[s], IDEAL_BIN) for s in SCALES],
                       zne_rem_fit, C["rem"], (10, 9)))

    for label, vals, fit, colour, dxy in curves:
        ax.plot(SCALES, vals, "o-", color=colour, linewidth=2, markersize=8,
                label=f"{label}  (measured)", zorder=3,
                markeredgecolor=SURFACE, markeredgewidth=1.5)
        if fit and fit.get("linear") is not None:
            slope = fit["linear_slope"]
            ax.plot(grid, fit["linear"] + slope * grid, "--", color=colour,
                    linewidth=1.4, alpha=0.75, zorder=2,
                    label=f"{label}  linear fit -> {fit['linear']:.3f}")
            ax.plot([0], [fit["linear"]], "D", color=colour, markersize=9,
                    markeredgecolor=SURFACE, markeredgewidth=1.5, zorder=4)
            # Offsets pull the two intercept labels apart: the raw and REM
            # fits land within ~0.09 of each other and would otherwise overlap.
            ax.annotate(f"{fit['linear']:.3f}", (0, fit["linear"]),
                        textcoords="offset points", xytext=dxy,
                        color=INK, fontsize=9, fontweight="bold")

    # Headroom reserved ABOVE the unphysical band so the legend has somewhere
    # to sit that is not on top of the data or the band label.
    top = max(1.30, ax.get_ylim()[1])
    ax.set_ylim(min(-0.06, ax.get_ylim()[0]), top)
    ax.axhspan(1.0, top, color="#e34948", alpha=0.07, zorder=0)
    ax.axhline(1.0, color="#e34948", linewidth=1, linestyle=":", zorder=1)
    ax.annotate("P > 1 is unphysical", (max(SCALES) * 0.06, 1.03),
                color="#a33", fontsize=9)
    ax.axvline(0, color="#8a8984", linewidth=0.8)
    ax.set_xlabel(r"noise scale  $\lambda$   (global folding $C \to C(C^\dagger C)^k$)")
    ax.set_ylabel("$P_{correct}$")
    ax.set_title("Zero-noise extrapolation")
    ax.set_xticks([0] + list(SCALES))
    ax.legend(loc="upper right", fontsize=9, ncol=2, columnspacing=1.2)
    fig.savefig(os.path.join(PLOTS_DIR, "qpe_zne_extrapolation.png"))
    plt.show()

    print(f"\n{'lambda':>8} " + " ".join(f"{c[0]:>10}" for c in curves))
    for i, s in enumerate(SCALES):
        print(f"{s:>8} " + " ".join(f"{c[1][i]:>10.4f}" for c in curves))
    print(f"{'0 (fit)':>8} " + " ".join(
        f"{(c[2] or {}).get('linear', float('nan')):>10.4f}" for c in curves))
else:
    print("Figure 2 skipped -- ZNE needs more than one lambda.")

In [ ]:
# -- Figure 3: the mitigation cells, with bootstrap intervals --------------
labels, vals, los, his, colours, valid = [], [], [], [], [], []
palette = {"raw (lambda=1)": C["raw"]}
for r in rows:
    key = r["condition"]
    colour = (C["raw"] if key.startswith("raw") else
              C["rem"] if key.startswith("REM") else
              C["zne_rem"] if key.startswith("ZNE+REM") else C["zne"])
    if r["p_correct"] is None:
        continue
    ci = r["ci"] or {}
    labels.append(key)
    vals.append(r["p_correct"])
    los.append(max(0.0, r["p_correct"] - (ci.get("lo") or r["p_correct"])))
    his.append(max(0.0, (ci.get("hi") or r["p_correct"]) - r["p_correct"]))
    colours.append(colour)
    valid.append(r["valid"])

y = np.arange(len(labels))[::-1]
fig, ax = plt.subplots(figsize=(8, 0.72 * len(labels) + 2.0))
bars = ax.barh(y, vals, height=0.62, color=colours, linewidth=0,
               xerr=[los, his], error_kw=dict(ecolor=INK_2, elinewidth=1.2, capsize=4))
for b, v, ok in zip(bars, vals, valid):
    ax.text(v + 0.012, b.get_y() + b.get_height() / 2,
            f"{v:.4f}" + ("" if ok else "  UNPHYSICAL"),
            va="center", fontsize=9, fontweight="bold",
            color=INK if ok else "#a33")
    if not ok:
        b.set_hatch("///")
        b.set_edgecolor("#a33")
        b.set_linewidth(1.0)

ax.axvline(1.0, color=C["ideal"], linewidth=1.5, linestyle="--")
ax.annotate("ideal = 1.0", (1.0, y.max() + 0.45), color=C["ideal"],
            fontsize=9, ha="center")
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.set_xlabel("$P_{correct}$   (95% bootstrap CI)")
ax.set_title("Mitigation cells")
ax.set_xlim(0, max(1.08, max(np.array(vals) + np.array(his)) * 1.15))
ax.grid(axis="y", visible=False)
fig.savefig(os.path.join(PLOTS_DIR, "qpe_mitigation_summary.png"))
plt.show()

## 10. Predicted vs actual — re-fitting the model constants

`rigetti_qpuf_common.py` carries placeholder gate times and fidelities;
Rigetti does not publish per-gate durations through Braket, and
`device_caps.json` came back with `specs: {}`, so `F1Q`/`F2Q` are guesses too.

`nativeQuilMetadata` is the authoritative post-compilation cost — what Quilc
*actually* ran. Compare it against what qiskit predicted at submission and
correct the constants before sizing the next reservation window.

Watch `topologicalSwaps` in particular: under verbatim it should be **0**. A
non-zero value means the circuit was re-routed, which for a ZNE run means the
folds may not have survived.

In [ ]:
def cmp_row(name, actual, predicted):
    if actual is None:
        return
    if isinstance(predicted, (int, float)) and predicted:
        ratio = actual / predicted if predicted else float("nan")
        print(f"    {name:<22} {actual:>12,.4g}   predicted {predicted:>12,.4g}   "
              f"ratio {ratio:>6.2f}x")
    else:
        print(f"    {name:<22} {actual:>12,.4g}")


any_meta = False
for s in SCALES:
    r = qpe_by_scale[s]
    nqm = r.get("native_quil_metadata")
    print(f"lambda = {s}   ({r.get('label', '')})")
    if not nqm:
        print("    no nativeQuilMetadata (simulator, or Rigetti returned none)\n")
        continue
    any_meta = True
    cmp_row("gateVolume", nqm.get("gateVolume"), r.get("n_gates"))
    cmp_row("gateDepth", nqm.get("gateDepth"), r.get("depth"))
    cmp_row("multiQubitGateDepth", nqm.get("multiQubitGateDepth"), r.get("depth_2q"))
    cmp_row("programFidelity", nqm.get("programFidelity"), r.get("est_fidelity"))
    swaps = nqm.get("topologicalSwaps")
    if swaps is not None:
        flag = "" if not (r.get("verbatim") and swaps) else "   <-- NON-ZERO UNDER VERBATIM"
        print(f"    {'topologicalSwaps':<22} {swaps:>12}{flag}")
    rt = nqm.get("qpuRuntimeEstimation")
    if rt is not None:
        cmp_row("runtime (s)", rt / 1000.0, r.get("est_runtime_parallel_s"))
    if r.get("task_time_seconds"):
        cmp_row("task wall time (s)", r["task_time_seconds"],
                r.get("est_runtime_parallel_s"))
    print()

if any_meta:
    fid = [(qpe_by_scale[s].get("native_quil_metadata") or {}).get("programFidelity")
           for s in SCALES]
    n2q = [qpe_by_scale[s].get("n_2q_gates") for s in SCALES]
    pairs = [(f, n) for f, n in zip(fid, n2q) if f and n]
    if pairs:
        f2q_implied = float(np.mean([f ** (1.0 / n) for f, n in pairs]))
        print(f"Implied per-CZ fidelity from programFidelity: {f2q_implied:.5f}")
        print(f"  -> consider setting F2Q = {f2q_implied:.5f} in "
              f"rigetti_qpuf_common.py (currently 0.97)")
        print("  Per MITIGATION_BRIEF section 7, ZNE is only usable at F2Q >~ 0.99;")
        print("  below that the model spread exceeds the quantity being estimated.")

## 11. Export

One JSON per analysis run, next to the results it summarises.

In [ ]:
summary = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "synthetic": bool(USE_SYNTHETIC),
    "run": {"n_prec": N_PREC, "n_targ": N_TARG, "state_mode": STATE_MODE,
            "phi": PHI, "ideal_bin": IDEAL_BIN, "scales": SCALES,
            "shots_per_task": sum(counts_by_scale[SCALES[0]].values())},
    "qubits": {"measured_qubits": MEASURED_QUBITS,
               "clbit_to_phys": CLBIT_TO_PHYS, "provenance": PROVENANCE},
    "batches": {"mode": BATCH_MODE, "provenance": BATCH_PROV,
                "ids": list(BATCHES), "anchors": anchors},
    "ddd": {"sequence": DDD_SEQ, "arms_present": sorted(siblings),
            "deltas": ddd_deltas},
    "rem": None if not HAS_REM else {
        "method": REM_METHOD, "condition_number": COND,
        "p_1_given_0": [float(m[1, 0]) for m in mats[:N_PREC]],
        "p_0_given_1": [float(m[0, 1]) for m in mats[:N_PREC]],
        "bootstrap": rem_boot},
    "zne": None if not HAS_ZNE else {"fit": zne_fit, "bootstrap": zne_boot},
    "zne_rem": None if not (HAS_ZNE and HAS_REM) else {
        "fit": zne_rem_fit, "bootstrap": zne_rem_boot},
    "table": rows,
    "task_arns": {r.get("label", r["job_id"]): r["job_id"] for r in fam},
}

name = ("analysis_summary_SYNTHETIC.json" if USE_SYNTHETIC
        else f"analysis_summary_nprec{N_PREC}_ntarg{N_TARG}_{STATE_MODE}"
             f"_ddd{DDD_SEQ}.json")
path = os.path.join(RESULTS_DIR if not USE_SYNTHETIC else PLOTS_DIR, name)
os.makedirs(os.path.dirname(path), exist_ok=True)
with open(path, "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"wrote {path}")

print("\n" + "=" * 72)
print("HEADLINE")
print("=" * 72)
for r in rows:
    if r["p_correct"] is None:
        continue
    tag = "" if r["valid"] else "   [UNPHYSICAL -- not a result]"
    print(f"  {r['condition']:<24} P_correct = {r['p_correct']:.4f}{tag}")
if HAS_REM and rem_boot:
    d = rem_boot["delta"]
    verdict = "significant" if (d["lo"] > 0 or d["hi"] < 0) else "not significant"
    print(f"\n  REM delta = {d['mean']:+.4f}  "
          f"[{d['lo']:+.4f}, {d['hi']:+.4f}]  ({verdict}, paired)")
print("=" * 72)